# Tune metadata models with separate patient checks: deduplicated Subset data

Does choosing model settings within training data improve predictions for held-out patients? We return to the spline and tree models in [05_metadata_models.ipynb](05_metadata_models.ipynb) and add the metadata NN from [06_metadata_neural_network.ipynb](06_metadata_neural_network.ipynb), using age, sex and anatomical site, the metadata alongside each image.

The computed results below compare fixed and selected versions of all four model families. The NN extension adds a prescribed 36-recipe search of hidden-layer size, learning rate and dropout. Results remain development evidence; the reserved test is not used.

For metric definitions, see the [short concept guide](../concepts.md).


**Scope and preservation:** this notebook uses the complete archived original as its section and visual template. Numerical findings are computed from the explicitly identified versioned run. The original experiments remain archived; no reserved-test labels or final all-development fit are used here.

**Experiment-scope change: version 2.** The completed AP-selected version 1 is preserved. This separately declared experiment selects settings by mean inner-fold ROC-AUC and reports mean outer-fold ROC-AUC as primary. AP, log loss and Brier are secondary; pooled summaries are supporting. The prior report’s heading structure, model colours, fonts and explanatory detail are retained. No all-development model or reserved-test result is produced.

## Context and methods

Tuning searches settings that control a model's flexibility. Selecting them on the same patients used to report performance could favour lucky choices. Nested cross-validation separates these roles with two levels of patient groups, or folds.

For each of five outer folds, reserve its patients for assessment. Use three preserved inner patient folds within the other patients, keeping each patient together. Compare settings on those inner groups, choose the highest mean AP and fit on all outer-training patients. Predict the outer group once. This evaluates the selection procedure, whose chosen settings can differ between folds.

The spline tries 18 combinations of regularisation, knots and degree. Regularisation restrains fitting; knots and degree control the age curve's shape. Each tree method tries 12 combinations. Every search includes the original settings, and all preprocessing is fitted within training.

Average precision (AP) is primary because it rewards ranking rare malignant images near the top. ROC-AUC measures how often a malignant image scores above a benign one; 0.5 is chance. Higher is better for both. Log loss and Brier score measure probability error; lower is better.

The outer folds have guided earlier model choices. Nested selection prevents direct use of their labels in their own search, but these remain development results rather than independent final-test evidence.


**Experiment-scope addition:** the NN now receives its own 36-recipe search, after the fixed NN comparison in stage 06. The selected method is the three-seed average, not the best individual seed. **Aggregation clarification:** mean outer-fold AP is the primary selection-procedure comparison; pooled scores and curves are secondary descriptive views.

## Data

### 1. Set paths and imports

Use the shared Subset fold file and project Python kernel. Output paths identify the saved tables and figures. Re-executing this report reads completed receipts and writes only presentation copies inside the new versioned run. The separate runner performs the searches; legacy named outputs are never overwritten.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/melanoma_detection").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from melanoma_detection.metadata_tuning_v2_reporting import (
    load_tuning, load_baseline_report, render_section, render_baseline_section,
    display_historical_cell, display_historical_source, original_snapshot,
)
plt.rcdefaults()

ORIGINAL_EXPORT_DPI = 320
NN_COMPARISON_COLORS = ['#2563EB', '#D97706', '#D97706', '#D97706', '#303030']
report = load_tuning(PROJECT_ROOT, 'dedup_v1', run_id='roc_auc_grid36_v1')
report["export_dpi"] = ORIGINAL_EXPORT_DPI
display(Markdown("**Version-2 saved-results report:** " + report["status"] +
    ". ROC-AUC is primary; AP and probability errors are secondary. No training, final fitting or reserved-test loading starts here."))


### 2. Load and check the development data

Confirm that the required fields and five folds are present and every patient's images stay in one outer fold. The counts below should match the earlier metadata notebooks, allowing us to compare the methods on the same data.


In [ ]:
render_section(report, 'inputs')

### Check the preserved inner patient groups

Compare fitting and inner-assessment patient counts, malignant support and overlap checks. The NN adds a third role: training-only stopping patients choose epoch count inside the inner fitting pool. This prevents the labels used to assess a candidate from also choosing its training duration. The clean condition filters original memberships; it does not rebuild folds after removing copies.

In [ ]:
render_section(report, 'tuning_memberships')

### 3. Define the model pipelines

Each pipeline includes both preprocessing and the model. A fresh training fit learns its own age transformation and category columns. This prevents validation patients from influencing the features used during their model's fitting.


In [ ]:
render_section(report, 'tuning_grid')

### Explain the neural-network settings being searched

The hidden layer combines the encoded age, sex and site inputs with learned weights. We compare 8, 16, 32 and 64 units. A larger layer can express more interactions, but has more parameters relative to the few malignant patients. A smaller layer may miss useful patterns; increasing size is therefore a question for held-out training groups, not an automatic improvement.

AdamW's learning rate controls the size of each weight update. Rates 0.0003, 0.001 and 0.003 compare slower, baseline and faster steps. Too small a rate may not reach a useful solution within the epoch cap; too large a rate may overshoot or fluctuate. Early stopping cannot fully replace this setting because it controls duration, rather than the update size.

Dropout randomly masks hidden activations during training. Rates 0, 0.2 and 0.4 compare no masking, the fixed baseline and stronger masking. It can reduce reliance on individual units but may also weaken a small network. Prediction always disables dropout. The Cartesian product gives 4 × 3 × 3 = 36 recipes, including the existing 32/0.001/0.2 procedure. Weight decay remains 0.001; depth, batch size 128, ReLU, unweighted binary cross-entropy and the epoch cap remain fixed.

For each candidate and inner assessment fold, separate inner-training patients choose duration through a stopping split. Preprocessing is learned only on fitting patients for that choice. A fresh preprocessor and network then refit on all inner-training patients for the selected duration and predict the disjoint inner assessment patients. The three prescribed seed predictions are averaged before AP is computed. The recipe with highest mean AP across the three inner assessment folds wins; deterministic recipe order resolves ties. A fresh selected procedure predicts outer patients once. The outer labels neither choose duration nor select settings.

The stopping split is prescribed rather than searched. For inner assessment group v, the remaining groups are [(v+1)%3, (v+2)%3]. Seed indices 0 and 2 stop on the first and fit on the second; index 1 reverses these roles. For example, assessing group 0 leaves groups 1 and 2: seeds 42 and 242 stop on group 1, while seed 142 stops on group 2. The fresh inner refit uses both groups before predicting group 0. Exact recipe ties prefer the current 32/0.001/0.2 setting, then ascending size, learning rate and dropout.

For the selected outer procedure, stopping fold = (outer fold + 1 + seed index) % 5. The training seed is base seed + outer fold for both duration selection and fresh refitting. These rules are fixed for all 36 candidates and both datasets; no outer score chooses a stopping group or a seed.

## Results

### 4. Train the fixed models and run the setting searches

For each outer fold, both versions train on the same patients and predict the same held-out patients. The tuned version also uses inner patient folds to select settings. This keeps the comparison focused on the selection procedure.


In [ ]:
render_section(report, 'tuning_candidates')

### 5. Check how often the same settings are selected

The first table records the choice within each outer-training group. The second counts how often the same combination wins. Similar choices across folds would suggest that the preferred settings are less sensitive to the training sample. Different choices can still work, but they give less support for adopting one fixed tuned combination.

The inner best AP is the score used to make a choice. It can favour a lucky setting and should not be read as the final performance estimate; use the outer scores below for that.


In [ ]:
render_section(report, 'tuning_choices')

### Read selection margins and training duration

The margin is selected mean inner ROC-AUC minus the runner-up's mean inner ROC-AUC. A small margin shows that settings were close in this training sample; it is not a confidence interval. Selected epochs differ by training partition and seed. Executed and reused fit counts make the extra search effort visible.

In [ ]:
render_section(report, 'tuning_duration')

### 6. Compare the held-out results

The first table labels equal-fold scores with primary ROC-AUC and secondary AP, plus supporting pooled scores separately. The second shows ROC-AUC changes within each fold. A positive "Tuned minus untuned" value favours tuning; a negative value favours the fixed model. Read the primary score, pooled secondary score and fold changes together so one favourable group does not hide inconsistent results.


In [ ]:
render_section(report, 'tuning_metrics')

### 7. Read the score comparison charts

The horizontal axis names model families. Blue bars use fixed settings; gold bars use settings selected within training. The first figure shows ROC-AUC vertically; the second shows AP. Higher bars are better, axes start at zero and annotations give exact scores.

Compare bars within each family, then compare the two measures. The direction and size of each change are calculated from the versioned predictions below; do not carry a historical direction into a new dataset. There are no uncertainty bars, so a small observed gain is not established as repeatable.


In [ ]:
render_section(report, 'tuning_bars')

### 8. Read the ROC and precision-recall curves

Each row shows one model family. Blue dashed lines use the fixed settings and gold solid lines use the settings selected within training data.

On the left, the horizontal axis is the fraction of benign images wrongly flagged and the vertical axis is the fraction of malignant images found (recall). A curve nearer the upper-left corner is preferable; the grey dotted diagonal marks chance-level ranking.

On the right, the horizontal axis is recall and the vertical axis is precision, the fraction of flagged images that are malignant. A higher curve at the same recall means fewer false alarms among flagged images. The grey dotted line marks the observed malignant-image rate. Precision is shown only from 0-30% to make these low values easier to compare.

The curves show where tuning helps or hurts across cutoffs. Crossing curves mean a version can perform better in one recall range and worse in another, even when one pooled score is higher.


The added fourth row is the metadata NN. Circle markers identify fixed settings and squares identify tuned settings, reinforcing the original blue/gold and dashed/solid encodings. Curves pool predictions and therefore complement the primary mean-fold scores.

In [ ]:
render_section(report, 'tuning_curves')

### 9. Collect the observed changes

The table puts pooled score changes beside the number of folds that improve, tie or worsen. Fold counts and score changes are computed for every family below. This is useful development evidence, but it does not establish a repeatable gain without an uncertainty check.


In [ ]:
render_section(report, 'tuning_changes')

### Interpret paired differences on the primary measure

The additional assessment resamples whole patients within each preserved outer fold, using identical draws for every compared model. Two thousand draws estimate 95% intervals for tuned NN minus fixed NN, fixed spline and tuned spline. These comparisons use mean-fold ROC-AUC as primary; secondary pooled results remain available in the assessment table. An interval spanning zero allows either direction under this assessment. It does not establish equivalence, nor account for all variability from retraining or expanding the search after earlier development results were examined. Read the paired comparison before treating a small score change as a reason to replace the final candidate.

In [ ]:
render_section(report, 'tuning_intervals')

## Limitations

The search covers only the settings listed here. It cannot establish the best possible spline, tree or neural-network model. Different inner folds could select different settings, especially with only 127 malignant images from 88 patients. The added NN paired intervals condition on saved predictions and omit retraining and broader adaptive-selection variability. Any small gain must be interpreted with these limits; an interval spanning zero leaves its direction unresolved.

The nested design keeps each outer patient's labels outside their own settings search. However, the same outer folds have already guided broader model and feature choices. These results remain development evidence, and nested tuning cannot undo that earlier reuse.

Ranking improvements do not prove that probabilities are calibrated or that a cutoff is useful in practice. This notebook does not establish clinical safety, causation or performance on patients from another hospital. The reserved test set has not been used to evaluate these models.


The NN grid keeps weight decay, batch size, activation and depth fixed. It evaluates one hidden-layer family; wider networks, alternative optimizers, class weighting and additional engineered features remain outside scope. Settings-selection frequency and runner-up margins describe training sensitivity, not new-patient confidence intervals.

## Summary

The computed summary below states the fixed-to-tuned changes and fold consistency for spline logistic, Random Forest, XGBoost and the metadata NN. Their searches include the fixed settings, but selecting the best inner score can still reduce outer performance through sampling variability. A favourable point estimate alone does not establish a dependable gain or calibrated probabilities.

Do not choose an individual seed or a global recipe from the outer results after inspecting them. Any final candidate change requires the same paired uncertainty and practical trade-off review.

Next, use notebook 08 to check uncertainty, probability calibration, subgroup results and training-selected cutoffs for the retained model. Evaluate the reserved test only after the complete method is frozen.


Freezing the procedure means recording the complete data cleaning, feature transformation, model or search recipe, seed averaging, stopping, calibration and threshold rules before final testing. The final model is then trained using development data according to those rules, and the reserved test is evaluated once. Changing a rule after seeing its test result would make that test part of development rather than an independent final check. That final-fit and test stage is not performed here.

In [ ]:
render_section(report, 'tuning_summary')